# Chains and cosmological constraints — `output/v1.txt`

Loads the CosmoSIS `emcee` chain produced by the HMF+MOR forecast pipeline,
inspects walker traces for burn-in/mixing, and makes a triangle (contour)
plot of the cosmological parameter constraints (`omega_m`, `sigma8_input`).

In [ ]:
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from getdist import MCSamples, plots

# must come after importing getdist.plots, which resets the mpl backend
%matplotlib inline

CHAIN_FILE = Path("../output/scatter_free_v1.txt")

## Load chain + header metadata

In [ ]:
def read_cosmosis_chain(path):
    """Parse a CosmoSIS text-sampler output file.

    Returns the data array, the column names, and a dict of the
    `#key=value` header metadata (sampler, walkers, nsteps, etc).
    """
    columns = None
    meta = {}
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):
                break
            if line.startswith("## "):
                continue
            header = line[1:].strip()
            if columns is None:
                columns = header.split()
                continue
            if "=" in header:
                key, _, val = header.partition("=")
                meta[key.strip()] = val.strip()

    data = np.loadtxt(path)
    return data, columns, meta


data, columns, meta = read_cosmosis_chain(CHAIN_FILE)
nwalkers = int(meta["walkers"])
nsteps = data.shape[0] // nwalkers

sampler_name = meta.get("sampler")
print(f"columns: {columns}")
print(f"sampler={sampler_name}  walkers={nwalkers}  steps={nsteps}  rows={data.shape[0]}")

## Reshape into (walker, step, param)

CosmoSIS's `emcee` sampler writes rows step-major: all walkers for step 0,
then all walkers for step 1, etc.

In [ ]:
param_names = [c for c in columns if c not in ("prior", "post")]
param_idx = [columns.index(p) for p in param_names]

chain = data.reshape(nsteps, nwalkers, len(columns)).transpose(1, 0, 2)
log_post = chain[:, :, columns.index("post")]

print(f"parameters: {param_names}")
print(f"chain shape (walkers, steps, cols): {chain.shape}")

## Walker traces

In [ ]:
fig, axes = plt.subplots(len(param_names) + 1, 1, figsize=(9, 2.2 * (len(param_names) + 1)), sharex=True)

for i, name in enumerate(param_names):
    ax = axes[i]
    for w in range(nwalkers):
        ax.plot(chain[w, :, param_idx[i]], color="C0", alpha=0.3, lw=0.7)
    ax.set_ylabel(name.split("--")[-1])

ax = axes[-1]
for w in range(nwalkers):
    ax.plot(log_post[w], color="C1", alpha=0.3, lw=0.7)
ax.set_ylabel("log-posterior")
ax.set_xlabel("step")

fig.suptitle("Walker traces — output/v1.txt", y=1.0)
fig.tight_layout()
plt.show()

## Burn-in cut

Adjust `burnin_frac` after looking at the traces above — the walkers should
look like a stationary "fuzzy caterpillar" past the burn-in point.

In [ ]:
burnin_frac = 0.5
burnin_steps = int(burnin_frac * nsteps)

flat_samples = chain[:, burnin_steps:, :].reshape(-1, len(columns))
flat_params = flat_samples[:, param_idx]
flat_post = flat_samples[:, columns.index("post")]

print(f"discarding first {burnin_steps}/{nsteps} steps as burn-in")
print(f"flattened samples: {flat_params.shape[0]}")

## Triangle plot of cosmological constraints

In [ ]:
labels = {
    "cosmological_parameters--omega_m": r"\Omega_m",
    "cosmological_parameters--sigma8_input": r"\sigma_8",
}
param_labels = [labels.get(p, p) for p in param_names]

samples = MCSamples(
    samples=flat_params,
    names=param_names,
    labels=param_labels,
    label="v1",
)

# fiducial values used to generate the mock data (see mass_function_like config)
fiducial = {
    "cosmological_parameters--omega_m": 0.318,
    "cosmological_parameters--sigma8_input": 0.80,
}
markers = {p: fiducial[p] for p in param_names if p in fiducial}

g = plots.get_subplot_plotter()
g.triangle_plot(samples, param_names, filled=True, markers=markers)
plt.show()

## 1D marginalized constraints

In [ ]:
for p, lbl in zip(param_names, param_labels):
    stats = samples.getMargeStats().parWithName(p)
    print(f"{lbl:>10s} = {stats.mean:.4f} +/- {stats.err:.4f}")

## Comparing `frac_scatter_model` assumptions

`scripts/submit_emcee_scatter_scan.slurm` refits the same mock data
(generated with `frac_scatter_data = 0.10` fixed) while scanning the
*assumed* model-side scatter, `frac_scatter_model`, away from that true
value. `v1.txt` itself is the matched case (`frac_scatter_model = 0.10`).
Overlaying the resulting constraints shows how a mismatch between the
assumed and true scatter biases and broadens `omega_m` / `sigma8_input`.

In [ ]:
def load_scatter_run(path, burnin_frac=0.5, label=None):
    """Read a chain file and return a burnin-cut getdist MCSamples."""
    data_, columns_, meta_ = read_cosmosis_chain(path)
    nwalkers_ = int(meta_["walkers"])
    nsteps_ = data_.shape[0] // nwalkers_
    chain_ = data_.reshape(nsteps_, nwalkers_, len(columns_)).transpose(1, 0, 2)
    burnin_steps_ = int(burnin_frac * nsteps_)

    pnames_ = [c for c in columns_ if c not in ("prior", "post")]
    pidx_ = [columns_.index(p) for p in pnames_]
    flat_ = chain_[:, burnin_steps_:, :].reshape(-1, len(columns_))

    return MCSamples(
        samples=flat_[:, pidx_],
        names=pnames_,
        labels=[labels.get(p, p) for p in pnames_],
        label=label,
    )


# (path, frac_scatter_model) — frac_scatter_data is fixed at 0.10 for all of them
scatter_runs = [
    ("../output/v1_sm00.txt", 0.00),
    ("../output/v1_sm05.txt", 0.05),
    ("../output/v1.txt", 0.10),
    ("../output/v1_sm20.txt", 0.20),
    ("../output/v1_sm40.txt", 0.40),
]

# categorical palette (fixed hue order, CVD-safe) — 5 runs need to be told apart
# at a glance, which a single-hue light->dark ramp does not give at thin contour
# linewidths, so identity beats magnitude-encoding here
scatter_colors = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7"]

scatter_samples = [
    load_scatter_run(path, label=f"frac_scatter_model = {frac:.2f}")
    for path, frac in scatter_runs
]

for (path, frac), s in zip(scatter_runs, scatter_samples):
    print(f"{path}: frac_scatter_model={frac:.2f}  samples={s.numrows}")

In [ ]:
g = plots.get_subplot_plotter()
g.triangle_plot(
    scatter_samples,
    param_names,
    filled=False,
    contour_colors=scatter_colors,
    markers=markers,
    legend_labels=[s.label for s in scatter_samples],
)
g.fig.suptitle("Cosmological constraints vs. assumed model scatter", y=1.02)
plt.show()

### Constraint shift vs. scatter mismatch

Mean ± 1σ for each parameter as a function of the assumed
`frac_scatter_model`, with the matched run (`frac_scatter_model =
frac_scatter_data = 0.10`) marked by the dashed vertical line and the
fiducial truth by the dashed horizontal line. Point colors match the runs
in the triangle plot above.

In [ ]:
fig, axes = plt.subplots(1, len(param_names), figsize=(5 * len(param_names), 4))

fracs = [frac for _, frac in scatter_runs]
for i, (p, lbl) in enumerate(zip(param_names, param_labels)):
    ax = axes[i]
    means = []
    errs = []
    for s in scatter_samples:
        stats = s.getMargeStats().parWithName(p)
        means.append(stats.mean)
        errs.append(stats.err)

    ax.plot(fracs, means, "-", color="0.75", lw=1, zorder=1)
    for frac, mean, err, color in zip(fracs, means, errs, scatter_colors):
        ax.errorbar(frac, mean, yerr=err, fmt="o", color=color, ecolor=color,
                     capsize=3, zorder=2)
    ax.axvline(0.10, color="0.6", ls="--", lw=1, label="matched (frac_scatter_data)")
    if p in fiducial:
        ax.axhline(fiducial[p], color="0.6", ls=":", lw=1, label="fiducial truth")
    ax.set_xlabel("frac_scatter_model")
    ax.set_ylabel(f"${lbl}$")
    ax.legend(fontsize=8)

fig.suptitle("Parameter constraints vs. assumed model scatter", y=1.02)
fig.tight_layout()
plt.show()